In [1]:
!pip install --no-index --find-links=/kaggle/input/ariel-2024-pqdm pqdm

import pandas as pd
import numpy as np

from tqdm import tqdm
from pqdm.threads import pqdm
import itertools

from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from scipy.signal import correlate, correlation_lags

from astropy.stats import sigma_clip
from scipy.signal import savgol_filter

import matplotlib.pyplot as plt

import time
__t0 = time.perf_counter()

class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"

    SCALE = 0.95
    SIGMA = 0.0009
    
    CUT_INF = 39
    CUT_SUP = 321
    
    # Cross-correlation fusion parameters
    FUSION_WINDOW = 50  # Window size for cross-correlation analysis
    FUSION_WEIGHT_DECAY = 0.1  # Exponential decay for lag weights
    FUSION_MIN_CORRELATION = 0.3  # Minimum correlation threshold
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 7
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

class CrossCorrelationFusion:
    """Implements cross-correlation based fusion for AIRS-CH0 and FGS1 signals"""
    
    def __init__(self, config):
        self.cfg = config
    
    def _compute_local_correlation(self, sig1, sig2, window_size):
        """Compute local cross-correlation between two signals using sliding windows"""
        correlations = []
        lags = []
        
        # Ensure signals have same length
        min_len = min(len(sig1), len(sig2))
        sig1, sig2 = sig1[:min_len], sig2[:min_len]
        
        # Sliding window cross-correlation
        for i in range(0, min_len - window_size + 1, window_size // 2):
            end_idx = min(i + window_size, min_len)
            window1 = sig1[i:end_idx]
            window2 = sig2[i:end_idx]
            
            # Skip if windows are too small or contain NaN
            if len(window1) < 10 or np.any(~np.isfinite(window1)) or np.any(~np.isfinite(window2)):
                correlations.append(0.0)
                lags.append(0)
                continue
            
            # Normalize windows
            window1 = (window1 - np.mean(window1)) / (np.std(window1) + 1e-8)
            window2 = (window2 - np.mean(window2)) / (np.std(window2) + 1e-8)
            
            # Cross-correlation
            correlation = correlate(window1, window2, mode='full')
            correlation_lags_arr = correlation_lags(len(window1), len(window2), mode='full')
            
            # Find peak correlation
            max_corr_idx = np.argmax(np.abs(correlation))
            max_correlation = correlation[max_corr_idx]
            optimal_lag = correlation_lags_arr[max_corr_idx]
            
            correlations.append(max_correlation)
            lags.append(optimal_lag)
        
        return np.array(correlations), np.array(lags)
    
    def _adaptive_weight_calculation(self, correlations, lags):
        """Calculate adaptive weights based on correlation strength and lag consistency"""
        # Base weight from correlation strength
        corr_weights = np.abs(correlations)
        corr_weights = np.clip(corr_weights, 0, 1)
        
        # Penalty for large lags (prefer synchronized signals)
        lag_weights = np.exp(-np.abs(lags) * self.cfg.FUSION_WEIGHT_DECAY)
        
        # Combined adaptive weights
        weights = corr_weights * lag_weights
        
        # Apply minimum correlation threshold
        weights = np.where(np.abs(correlations) >= self.cfg.FUSION_MIN_CORRELATION, weights, 0.1)
        
        return weights
    
    def _interpolate_weights(self, weights, original_length):
        """Interpolate weights to match original signal length"""
        if len(weights) == 0:
            return np.ones(original_length) * 0.5
        
        # Create interpolation indices
        weight_indices = np.linspace(0, original_length - 1, len(weights))
        full_indices = np.arange(original_length)
        
        # Interpolate weights
        interpolated = np.interp(full_indices, weight_indices, weights)
        
        # Ensure weights are normalized between 0 and 1
        interpolated = np.clip(interpolated, 0.1, 0.9)
        
        return interpolated
    
    def fuse_signals(self, fgs_signal, airs_signal):
        """
        Fuse FGS1 and AIRS-CH0 signals using cross-correlation analysis
        
        Args:
            fgs_signal: FGS1 signal (1D)
            airs_signal: AIRS-CH0 signal (typically 2D, will use mean)
        
        Returns:
            fused_signal: Combined signal using cross-correlation weights
        """
        # Handle AIRS signal dimensionality
        if airs_signal.ndim > 1:
            airs_1d = np.nanmean(airs_signal, axis=1)
        else:
            airs_1d = airs_signal
        
        # Ensure same length
        min_len = min(len(fgs_signal), len(airs_1d))
        fgs_1d = fgs_signal[:min_len]
        airs_1d = airs_1d[:min_len]
        
        # Handle edge cases
        if min_len < self.cfg.FUSION_WINDOW:
            # If signals too short, use simple average
            return (fgs_1d + airs_1d) / 2
        
        # Compute local cross-correlations
        correlations, lags = self._compute_local_correlation(
            fgs_1d, airs_1d, self.cfg.FUSION_WINDOW
        )
        
        # Calculate adaptive weights
        local_weights = self._adaptive_weight_calculation(correlations, lags)
        
        # Interpolate weights to full signal length
        fgs_weights = self._interpolate_weights(local_weights, min_len)
        airs_weights = 1 - fgs_weights
        
        # Apply lag compensation if significant consistent lag detected
        median_lag = np.median(lags[np.abs(correlations) >= self.cfg.FUSION_MIN_CORRELATION])
        if np.isfinite(median_lag) and np.abs(median_lag) > 0:
            lag = int(np.round(median_lag))
            if lag > 0:
                # AIRS leads FGS
                airs_aligned = airs_1d[:-lag] if lag < len(airs_1d) else airs_1d
                fgs_aligned = fgs_1d[lag:] if lag < len(fgs_1d) else fgs_1d
            elif lag < 0:
                # FGS leads AIRS
                lag = -lag
                fgs_aligned = fgs_1d[:-lag] if lag < len(fgs_1d) else fgs_1d
                airs_aligned = airs_1d[lag:] if lag < len(airs_1d) else airs_1d
            else:
                fgs_aligned, airs_aligned = fgs_1d, airs_1d
            
            # Adjust weights for aligned signals
            min_aligned_len = min(len(fgs_aligned), len(airs_aligned))
            fgs_weights = fgs_weights[:min_aligned_len]
            airs_weights = airs_weights[:min_aligned_len]
            fgs_aligned = fgs_aligned[:min_aligned_len]
            airs_aligned = airs_aligned[:min_aligned_len]
        else:
            fgs_aligned, airs_aligned = fgs_1d, airs_1d
        
        # Weighted fusion
        fused = fgs_weights * fgs_aligned + airs_weights * airs_aligned
        
        return fused

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)
        self.fusion = CrossCorrelationFusion(config)

    def _apply_linear_corr(self, linear_corr, signal):
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]

        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]

        signal = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet").to_numpy()
        dark = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet").to_numpy()
        dead = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet").to_numpy()
        flat = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet").to_numpy()
        linear_corr = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet").values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset

        hot = sigma_clip(dark, sigma=5, maxiters=5).mask

        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]

        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]

        np.maximum(signal, 0, out=signal)

        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)

        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment

        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        mean_signal = np.nanmean(signal_roi, axis=1)

        cds_signal = mean_signal[1::2] - mean_signal[0::2]

        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var

            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]
        return binned

    def _process_planet_sensor(self, args):
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def _fuse_sensor_data(self, fgs_data, airs_data):
        """
        Fuse FGS1 and AIRS-CH0 data using cross-correlation based fusion
        
        Args:
            fgs_data: List of preprocessed FGS1 signals
            airs_data: List of preprocessed AIRS-CH0 signals
        
        Returns:
            fused_data: Array of fused signals with enhanced transit detection
        """
        fused_signals = []
        
        for fgs_signal, airs_signal in zip(fgs_data, airs_data):
            # Extract 1D signals
            fgs_1d = fgs_signal[:, 0]  # FGS1 is already 1D after preprocessing
            airs_1d = np.nanmean(airs_signal, axis=1)  # Average AIRS channels
            
            # Apply cross-correlation fusion
            fused = self.fusion.fuse_signals(fgs_1d, airs_1d)
            
            # Create combined signal array with original AIRS channels plus fused signal
            # Shape: [time_steps, airs_channels + 1]
            combined = np.column_stack([
                fused,  # Fused signal as primary channel
                airs_signal  # Original AIRS channels as supplementary
            ])
            
            fused_signals.append(combined)
        
        return np.stack(fused_signals)

    def process_all_data(self):
        # Process individual sensors
        args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
        preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
        preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        # Apply cross-correlation based fusion instead of simple concatenation
        fused_signal = self._fuse_sensor_data(preprocessed_fgs1, preprocessed_airs_ch0)
        
        return fused_signal

def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    if g1_max != 0: g1 /= g1_max
    if g2_max != 0: g2 /= g2_max
    phase1 = int(np.argmin(g1)); phase2 = int(np.argmax(g2)) + min_idx
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    for single in preprocessed_data:
        # Use the fused signal (first channel) instead of separate FGS
        fused_signal = single[:, 0]
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        oot = (fused_signal[: p1 - delta] if p1 - delta > 0 else np.empty(0, fused_signal.dtype))
        if p2 + delta < fused_signal.size:
            oot = np.concatenate([oot, fused_signal[p2 + delta :]])
        inn = fused_signal[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fused_signal))
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.25)

    return k * cfg.SIGMA

def estimate_sigma_air(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        white = np.nanmean(single[:, 1:], axis=1)
        white_s = savgol_filter(white, 20, 2)

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.90, 1.20)

    return k * cfg.SIGMA

class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index

        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        
        return error

    def predict(self, single_preprocessed_signal):
        # Use the fused signal (first channel) as primary signal for transit detection
        signal_1d = single_preprocessed_signal[:, 0]  # Fused signal
        
        # Apply smoothing
        signal_1d = savgol_filter(signal_1d, 20, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        phase2 = min(len(signal_1d) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)    

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
        ]
        return np.array(predictions) * self.cfg.SCALE

class SubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions, sigma_fgs=None, sigma_air=None):
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        submission_df.to_csv("submission.csv")
        return submission_df


# Main execution
config = Config()
    
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

submission_generator = SubmissionGenerator(config)
submission = submission_generator.create(predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)

__t1 = time.perf_counter()
elapsed = __t1 - __t0
print(f"[TIMING] total runtime: {elapsed:.2f} s ({elapsed/60:.2f} min)")
print(f"[INFO] Cross-correlation fusion applied to {len(preprocessed_data)} planet signals")

Looking in links: /kaggle/input/ariel-2024-pqdm
Processing /kaggle/input/ariel-2024-pqdm/pqdm-0.2.0-py2.py3-none-any.whl
Processing /kaggle/input/ariel-2024-pqdm/bounded_pool_executor-0.0.3-py3-none-any.whl (from pqdm)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  9.29it/s]

[TIMING] total runtime: 5.41 s (0.09 min)
[INFO] Cross-correlation fusion applied to 1 planet signals
